# The consumer boundary: the data contract and the five analytics entry points

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.facade`

**Modules covered** `facade.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The boundary is the whole of what a consumer of this engine adopts: the data contract it satisfies, and the five entry points it then calls: factor exposures, risk models, construction, evaluation, and attribution with the Euler risk budget. The module holds no arithmetic of its own. It binds each entry point to the layer modules that do the work, so the names a consumer imports are the modules this repository runs and not a second copy of them, and its entry point prints that inventory rather than a report of results.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `facade.py::main` traces to the consumer boundary of this build, established rather than cited: the data contract a consumer satisfies and the five analytics entry points it then calls, named in one place so a layer reshuffle costs an edit here rather than an edit in every consumer

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`facade.py`**

The engine's consumer boundary: the data contract and the five analytics entry points, in one import.

A consumer of this package satisfies the table contract (one row per instrument-month carrying
`period_month` and `available_from`, plus a manifest) and then calls the analytics through the names
below rather than through the layer modules directly. The indirection is not for its own sake: a module
moved inside a layer, a function renamed or a layer split then costs one edit in this file instead of an
edit in every consumer, and a consumer that pins the revision by tag gets a boundary whose contents it
can name before it imports anything.

**The five groups are the order a portfolio build exercises them in**: exposures, risk model,
construction, evaluation, and the decomposition of what came out. The Euler risk budget is grouped with
attribution rather than given a sixth name because the two read the same realised series into named
contributions, so a consumer that runs one almost always runs the other.

**What is deliberately absent.** No function is wrapped: a wrapper duplicates a signature that then
drifts from the module's own, or becomes a second place for a convention to be stated, and the module
headers are where those conventions live. No window, universe or mandate is applied either. Those are
this build's own decisions, made for one panel and one mandate, and a consumer with a different mandate
supplies its own frames and its own policy weights. The boundary carries the machinery, not the choices.

**Version.** The repository tags the commit this boundary was released at, and `VERSION` names that
revision so a consumer that pinned the tag can read back which one it imported. The prose deliverables
carry the same constant rather than one of their own, so a document cannot cite a version the boundary
does not have.

## 3. The data contract it consumes, and the as-of rule

The contract is the engine's own, unchanged by the boundary: one row per instrument-month carrying `period_month` and `available_from`, a manifest the loader verifies and fails closed on, and the as-of rule that a bar becomes readable on the first day of the month after the month it is labelled with. Nothing optional is added to it here, and no default is applied on the consumer's behalf: the window, the universe, the policy weights and the constraint set are this build's decisions for one panel and one mandate, so a consumer with a different mandate passes its own frames and its own weights into the entry points rather than inheriting these.

## 4. The worked example on small numbers, with the identity checked

The boundary has no arithmetic, so the identity worth checking is object identity: a group member must be the layer's own module rather than a copy, which is what makes the promise that a module moved inside a layer costs one edit here rather than an edit in every consumer. The cell checks that, and checks that the groups are the five entry points and the contract the module declares.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
from portfolio_workbench import facade
from portfolio_workbench.budget import euler
from portfolio_workbench.risk import covariance

declared = {"contract", "factor_exposures", "risk_models", "construction", "evaluation", "attribution"}
assert {name for name, _ in facade.GROUPS} == declared

# Identity, not equality: a copy of a module would be the second account of it that this design
# exists to avoid.
assert facade.attribution.euler is euler
assert facade.risk_models.covariance is covariance
assert facade.attribution.euler.__name__ == "portfolio_workbench.budget.euler"

calls = sum(len(vars(group)) for _, group in facade.GROUPS)
print(f"{len(facade.GROUPS)} groups carrying {calls} modules, version {facade.VERSION}")
print("identity holds for every member: " + ", ".join(
    f"{name} {len(vars(group))}" for name, group in facade.GROUPS))

6 groups carrying 19 modules, version 1.1
identity holds for every member: contract 5, factor_exposures 4, risk_models 1, construction 3, evaluation 3, attribution 3


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench import facade

print(f"build version {facade.VERSION}, the revision the repository tags this commit at")
print("entry points: " + ", ".join(name for name, _ in facade.GROUPS))
print("no window, universe, policy weights or constraint set is applied by the boundary")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
build version 1.1, the revision the repository tags this commit at
entry points: contract, factor_exposures, risk_models, construction, evaluation, attribution
no window, universe, policy weights or constraint set is applied by the boundary


In [3]:
import subprocess
import sys
from pathlib import Path

# The package is imported from the repository root, so the run needs the root as its working
# directory rather than wherever the kernel was started. Walked up from the kernel's own directory
# rather than written in at generation time: an absolute path here would name one workstation, and
# the notebook is a file every reader runs on their own.
root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "portfolio_workbench").is_dir()
)

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.facade"], capture_output=True, text=True, cwd=root
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[table] consumer boundary version 1.1, 5 analytics entry points over the data contract
[table] contract: loader, manifest, panel, quality, sql
[table] factor_exposures: components, exposures, spanning, spine
[table] risk_models: covariance
[table] construction: constraints, families, means
[table] evaluation: metrics, statistics, walkforward
[table] attribution: brinson, euler, factor



## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The boundary is the surface a consumer pins: a revision tagged in this repository, with the contract above it and five named entry points behind it. A reader must not read the grouping as a completeness claim about portfolio methods. It is the order this build exercises them in, and the risk budget sits inside the attribution group because the two decompose the same realised series. A reader must not read the version constant as a guarantee either: the repository tag pins the commit, and a consumer that imports the layer modules directly rather than through these names has pinned nothing.

## 7. What this module does not establish

Nothing here establishes that a consumer satisfies the contract by importing this module: the contract is a table shape, and whether another project's files carry it is decided by its own run of the quality gate and by comparing its returns with the pandas path. Nothing here establishes that the five groups are the five a different mandate needs, and nothing here makes any claim about results: the numbers live in the modules behind these names, each with its own notebook stating what it does not establish.